In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import DeltaTable

dbutils.widgets.text("incremntal_flag",'0')
incremental_flag = dbutils.widgets.get("incremntal_flag")

In [0]:
df = spark.sql("select * from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`")
df.display()

In [0]:
df_src = spark.sql("select distinct Model_ID,Product_Name,model_category as Model_Category from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`")
df_src.display()



In [0]:
if spark.catalog.tableExists("car_sales.gold.dim_model"):
    df_sink = spark.sql("select Model_ID,Product_Name,Model_Category,dim_model_key from car_sales.gold.dim_model")
else:
    df_sink = spark.sql("select Model_ID,Product_Name,model_category as Model_Category, 0 dim_model_key from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver` where 1=0")
df_sink.display()

In [0]:
df_filter = df_src.join(df_sink,df_src.Model_ID == df_sink.Model_ID,'left').select(df_src.Model_ID,df_src.Product_Name,df_src.Model_Category,df_sink.dim_model_key)

df_filter_old = df_filter.where(df_filter['dim_model_key'].isNotNull())
df_filter_new = df_filter.where(df_filter['dim_model_key'].isNull())

if (incremental_flag == "0"):
    max_value = 1
else:
    if spark.catalog.tableExists("car_sales.gold.dim_model"):
        df = spark.sql("select max(dim_model_key) from car_sales.gold.dim_model")
        max_value = df.collect()[0][0]

df_filter_new = df_filter_new.withColumn("dim_model_key",F.lit(max_value) + F.monotonically_increasing_id())

df_final = df_filter_old.union(df_filter_new)

if spark.catalog.tableExists("car_sales.gold.dim_model"):
    delta_table = DeltaTable.forPath(spark,"abfss://carsales@adls7786.dfs.core.windows.net/gold/dim_model")
    delta_table.alias('t').merge(df_final.alias('s'),"t.dim_model_key = s.dim_model_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format('delta').mode('overwrite').option("path",'abfss://carsales@adls7786.dfs.core.windows.net/gold/dim_model').saveAsTable("car_sales.gold.dim_model")


In [0]:
%sql

select * from car_sales.gold.dim_model;